### For testing the caching directory ("out_dir") of the boltz spec
-> Run with GPU allocation

In [ ]:
from flexcraft.structure.boltz import *
from flexcraft.data.data import DesignData
from flexcraft.files import PDBFile
from flexcraft.sequence.aa_codes import *
from flexcraft.utils.rng import Keygen
from pathlib import Path

In [ ]:
test_dir = Path("/home/hgf_dkfz/hgf_dsb0249/workspaces/haicwork/hgf_dsb0249-BinderDesign/flexcraft/data/boltzCaching/test_dir")
test_file = test_dir/"test_template_clean.pdb"

test_structure = PDBFile(path=test_file).to_data()
test_seq = test_structure.to_sequence_string()
template_file = test_file

In [ ]:
config = {
            "docking":False,
            "redocking":False,
            "parameter_path":Path("./params/boltz"),
            "model_name":"boltz2_conf",
            "num_recycle":2,
            "num_samples":1,
            "num_sampling_steps":25,
            "deterministic":False,
            "predictor":None,
            "msa":True,
            }
boltz = Joltz2(model=config["model_name"]+".ckpt", cache=config["parameter_path"])


In [ ]:
boltz_evaluator, config["params"] = boltz.evaluator(
    num_recycle=config["num_recycle"],
    num_sampling_steps=config["num_sampling_steps"],
    deterministic=config["deterministic"]
)
def _wrap_eval(key, params, joltz_input, num_samples=config["num_samples"]):
    return boltz_evaluator(params
    ).predict(key, joltz_input, num_samples=num_samples,)
predictor = jax.jit(_wrap_eval, static_argnames="num_samples")

In [ ]:
key = Keygen(11)

In [ ]:
caching_dir = test_dir/"boltz_cache"
caching_dir.mkdir(exist_ok=True)

joltz_spec = JoltzSpec(out_dir=caching_dir).add_protein(test_seq, use_msa=config["msa"])
joltz_spec = joltz_spec.add_template(template_file.__str__())

boltz_input, boltz_writer = joltz_spec.to_input(pad=True, cache=config["parameter_path"])
boltz_result = predictor(
    key(),
    config["params"],
    boltz_input,
    num_samples=config["num_samples"]
    )

In [ ]:
# if msa is not recomputed and "All inputs are already processed." gets printed, we can assume the cache is working correctly
joltz_spec = JoltzSpec(out_dir=caching_dir).add_protein(test_seq, use_msa=config["msa"])
joltz_spec = joltz_spec.add_template(template_file.__str__())

boltz_input, boltz_writer = joltz_spec.to_input(pad=True, cache=config["parameter_path"])
boltz_result_cached = predictor(
    key(),
    config["params"],
    boltz_input,
    num_samples=config["num_samples"]
    )